# TP : Couches d'un datalake

Ce TP propose de reproduire localement l'organisation de données en couche au sein d'un datalake.

Rappel du modèle en couches présenté en cours (architecture "_medallion_"):

```
    Données entrantes => [Landing Zone] => [🥉 Bronze] => [🥈 Silver] => [🥇 Gold]
```

avec :
- _Landing Zone_ = espace de réception
- _Bronze_ = données brutes, non modifiées
- _Silver_ = données avec des modifications techniques de base
- _Gold_ = données raffinées avec des règles métier

Les couches sont simulées par des répertoires. On fournit la structure de répertoires suivante :

```
data
 ├─ landing
 │   ├─ diameters.zip
 │   └─ referentials.zip
 ├─ bronze
 │   ├─ diameters
 │   └─ referentials
 ├─ silver
 │   ├─ diameters
 │   └─ referentials
 └─ gold
```

Les fichiers ZIP dans `landing` sont donc les données déjà posées dans la _landing zone_.

Les répertoires `bronze` et `silver` contiennent chacun 2 sous-répertoires vides : `diameters` et `referentials`, un par source de données. On veillera dans la suite à créer les fichiers dans le bon sous-répertoire, selon la nature des données manipulées (référentiels ou diamètres). Il ne doit pas y avoir de fichiers directement dans `bronze` ou `silver`.

Le répertoire `gold` est vide au départ. Il va fusionner les sources donc pas besoin de sous-répertoire.

In [ ]:
# Réglez la variable `DATA_DIR` avec l'emplacement où vous avez dézippé les données fournies :
# chemin du répertoire `data`
DATA_DIR = 'data'

In [ ]:
# On prépare les chemins vers les répertoires de couches

import os

LANDING_DIR = os.path.join(DATA_DIR, 'landing')
BRONZE_DIR = os.path.join(DATA_DIR, 'bronze')
SILVER_DIR = os.path.join(DATA_DIR, 'silver')
GOLD_DIR = os.path.join(DATA_DIR, 'gold')

# Exemples d'utilisation :
# Chemin vers un fichier de réferentiel dans la couche "Silver" : os.path.join(SILVER_DIR, 'referentials', 'varieties.parquet')
# Chemin vers un fichier dans la couche "Gold" : os.path.join(GOLD_DIR, 'gold_ok.parquet')

In [ ]:
import pandas as pd

## Passage vers la couche _Bronze_

On se contente de dézipper les fichiers sources vers le répertoire `bronze`.

In [ ]:
import os
from zipfile import ZipFile

Voici le squelette de code pour les diamètres (voir les exemples ci-dessus pour la gestion des chemins) :

In [ ]:
with ZipFile(os.path.join(LANDING_DIR, 'diameters.zip')) as landing_diameters:
    landing_diameters.extractall("""CHANGE ME""")

Faire de même pour les référentiels :

Vérifier dans le répertoire `bronze` que les données sont bien présentes.

## Passage vers la couche _Silver_

### Référentiels

Pour les référentiels, on fait confiance à la source et on se contentera de sauver les données à l'identique au format Parquet.

Charger les données de la couche _Bronze_ en mémoire avec Pandas, et les sauver sans modification au format Parquet (un fichier, pas de partitionnement, ne pas écrire l'index Pandas) dans le répertoire `silver/referentials`.

Vérifier que les données sont correctement chargées (attention au séparateur de champs CSV).

### Pour les diamètres

La qualité des données est variable et il va falloir les remettre d'aplomb.

Charger **toutes** les données de la couche _Bronze_ dans un seul gros dataframe, et l'explorer pour avoir une idée de la situation.

In [10]:
bronze_diameters

,timestamp,variety_id,diameter,detection_status
0,2026-01-10 14:52:03,12,10.305,Ok
1,2026-01-10 14:53:00,7,8.995,yes
2,2026-01-10 14:54:05,12,10.099,Fail
3,2026-01-10 14:55:16,12,9.115,ERror
4,2026-01-10 14:56:12,16,9.875,FAILURE
...,...,...,...,...
35,2026-01-10 16:47:12,6,10.804,OK
36,2026-01-10 16:48:16,1,8.363,ERror
37,2026-01-10 16:49:15,2,10.880,Fail
38,2026-01-10 16:50:07,12,10.681,GOod


In [ ]:
# ... continuer l'exploration des différents champs pour guider la question suivante

Créer un nouveau dataframe qui contient les données mises en forme et corrigées :
- colonne `timestamp` de type "datetime" Pandas
- élimination des diamètres invalides
- remplacement de la colonne `detection_status` par un entier pouvant prendre les valeurs 1 (état OK) ou 0 (état défaillant)

Sauver ensuite ce nouveau dataframe au format Parquet dans le répertoire `silver/diameters` (pas un fichier, pas de partitionnement, pas d'index).

## Passage vers la couche _Gold_

Recharger séparément les 2 dataframes du répertoire `silver`, et en déduire 2 nouveaux dataframes :
- `gold_ok` qui contient la jointure entre les diamètres et les référentiels
- `gold_error`un qui contient les données de diamètre (uniquement) n'ayant pas pu être jointes aux variétés

Sauver ces 2 fichiers dans le répertoire `gold` (format Parquet, pas de partition). Vous pouvez nommer les fichiers comme les dataframes (ex. `gold_ok.parquet`).